<a href="https://colab.research.google.com/github/MBKNgcobo/GF.github.io/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data loading

In [ ]:
!pip install spacy spacytextblob
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
import pandas as pd
import nltk
import re
import string
import pandas as pd
import csv

from nltk.corpus import stopwords

# Ensure spacytextblob is installed and available
!pip install spacytextblob
import spacy
import pandas as pd
from spacytextblob.spacytextblob import SpacyTextBlob

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'consumer-reviews-of-amazon-products' dataset.
Path to dataset files: /kaggle/input/consumer-reviews-of-amazon-products


In [ ]:
import os

# The 'path' variable from kagglehub.dataset_download is a directory.
# We need to find the specific CSV file within this directory.
# Common name for this dataset's main CSV file:
csv_file_name = 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv'

# Construct the full path to the CSV file
full_csv_path = os.path.join(path, csv_file_name)

# Try with the Python engine and skip broken lines (pandas >= 1.3)
df = pd.read_csv(full_csv_path,
                 engine='python',
                 on_bad_lines='skip',
                 quoting=csv.QUOTE_MINIMAL,
                 encoding='utf-8')
df.shape
df.head()

,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.dateSeen,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,"2018-05-27T00:00:00Z,2017-09-18T00:00:00Z,2017...",False,NaN,0,3,http://reviews.bestbuy.com/3545/5442403/review...,I thought it would be as big as small paper bu...,Too small,llyyue,https://www.newegg.com/Product/Product.aspx%25...
1,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,"2018-05-27T00:00:00Z,2017-07-07T00:00:00Z,2017...",True,NaN,0,5,http://reviews.bestbuy.com/3545/5442403/review...,This kindle is light and easy to use especiall...,Great light reader. Easy to use at the beach,Charmi,https://www.newegg.com/Product/Product.aspx%25...
2,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-05-27T00:00:00Z,True,NaN,0,4,https://reviews.bestbuy.com/3545/5442403/revie...,Didnt know how much i'd use a kindle so went f...,Great for the price,johnnyjojojo,https://www.newegg.com/Product/Product.aspx%25...
3,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-10-09T00:00:00Z,True,177283626.0,3,5,https://redsky.target.com/groot-domain-api/v1/...,I am 100 happy with my purchase. I caught it o...,A Great Buy,Kdperry,https://www.newegg.com/Product/Product.aspx%25...
4,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-05-27T00:00:00Z,True,NaN,0,5,https://reviews.bestbuy.com/3545/5442403/revie...,Solid entry level Kindle. Great for kids. Gift...,Solid entry-level Kindle. Great for kids,Johnnyblack,https://www.newegg.com/Product/Product.aspx%25...


#Feature engineering

In [ ]:
# Keep only required columns
df = df[['reviews.text', 'reviews.rating']].copy()

# Drop missing values
df.dropna(subset=['reviews.text', 'reviews.rating'], inplace=True)

# Convert ratings to int (some datasets store as float)
df['reviews.rating'] = df['reviews.rating'].astype(int)


In [ ]:
def rating_to_sentiment(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

df['label'] = df['reviews.rating'].apply(rating_to_sentiment)


In [ ]:
# Download stopwords
nltk.download('stopwords')

STOP_WORDS = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """
    Clean product review text for sentiment analysis and LSTM models.
    """
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords
    text = " ".join(
        word for word in text.split() if word not in STOP_WORDS
    )

    return text


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df['clean_text'] = df['reviews.text'].apply(clean_text)


In [ ]:
df['label'].value_counts()


,count
label,
positive,4686
neutral,197
negative,117


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df['clean_text']
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# If ANY class has <2 samples, remove it
min_class_count = y_encoded.value_counts().min() if hasattr(y_encoded, "value_counts") else min(pd.Series(y_encoded).value_counts())
print("Min class count:", min_class_count)


Min class count: 117


In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


#Model design

In [ ]:
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob

# Load the English spaCy model
nlp = spacy.load("en_core_web_sm")

# Add the SpacyTextBlob pipe to the nlp pipeline
if "spacytextblob" not in nlp.pipe_names:
    nlp.add_pipe("spacytextblob")

print("spacytextblob successfully added to the spaCy pipeline.")

spacytextblob successfully added to the spaCy pipeline.


In [ ]:
from typing import Callable

# Analyze sentiment function
def analyze_sentiment(
    nlp,
    text: str,
    clean_fn: Callable[[str], str] | None = None,
    threshold_pos: float = 0.1,
    threshold_neg: float = -0.1,
    max_length: int = 10000,
    silent: bool = False
) -> dict:
    """
    Analyze sentiment for a single text using spaCy + spacy-textblob.

    Args:
      nlp: spaCy language pipeline with spacytextblob added (doc._.blob)
      text: input review (string). If empty/invalid returns neutral defaults.
      clean_fn: optional callable to clean text (e.g. your clean_text). If None,
                the function will use the text as-is.
      threshold_pos: polarity > threshold_pos => "positive"
      threshold_neg: polarity < threshold_neg => "negative"
      max_length: truncate very long texts to this many characters (prevents pipeline issues)
      silent: if False, will include 'error' info in returned dict on exceptions

    Returns:
      dict containing:
        polarity (float), subjectivity (float), label ('positive'|'neutral'|'negative'),
        raw_text (original), cleaned_text (used for analysis), maybe 'error' if exception.
    """
    # Basic validation
    if not isinstance(text, str) or text.strip() == "":
        return {
            "polarity": 0.0,
            "subjectivity": 0.0,
            "label": "neutral",
            "raw_text": text,
            "cleaned_text": ""
        }

    try:
        # Apply cleaning function
        cleaned = clean_text(text) if callable(clean_text) else text

        # Ensure cleaned is a string
        if not isinstance(cleaned, str):
            cleaned = str(cleaned)

        cleaned = cleaned.strip()

        # Remove excessive whitespace and control chars
        cleaned = re.sub(r"\s+", " ", cleaned)

        # Truncate to avoid extremely long inputs breaking the pipeline
        if len(cleaned) > max_length:
            cleaned = cleaned[:max_length]

        # fallback if cleaning removed everything
        text_for_analysis = cleaned if cleaned else text

        # run through spaCy
        doc = nlp(text_for_analysis)

        # Access spacytextblob scores
        polarity = None
        subjectivity = None

        # spacy-textblob provides doc._.polarity & doc._.subjectivity OR doc._.blob.polarity
        polarity = getattr(doc._, "polarity", None)
        subjectivity = getattr(doc._, "subjectivity", None)

        if polarity is None or subjectivity is None:
            blob = getattr(doc._, "blob", None)
            if blob is not None:
                polarity = getattr(blob, "polarity", 0.0) if polarity is None else polarity
                subjectivity = getattr(blob, "subjectivity", 0.0) if subjectivity is None else subjectivity

        # final fallbacks
        polarity = float(polarity) if polarity is not None else 0.0
        subjectivity = float(subjectivity) if subjectivity is not None else 0.0

        # map to discrete label
        if polarity > threshold_pos:
            label = "positive"
        elif polarity < threshold_neg:
            label = "negative"
        else:
            label = "neutral"

        return {
            "polarity": polarity,
            "subjectivity": subjectivity,
            "label": label,
            "raw_text": text,
            "cleaned_text": cleaned
        }

    except Exception as e:
        # On exception, return neutral defaults and include error message unless silent
        out = {
            "polarity": 0.0,
            "subjectivity": 0.0,
            "label": "neutral",
            "raw_text": text,
            "cleaned_text": cleaned if 'cleaned' in locals() else ""
        }
        if not silent:
            out["error"] = str(e)
        return out

In [ ]:

from tqdm import tqdm
tqdm.pandas()  # enables progress_apply

# Run analyzer over the column and expand the dicts into a DataFrame
sentiment_series = df['clean_text'].progress_apply(lambda t: analyze_sentiment(nlp, t, clean_fn=None))

# Convert Series[dict] into a DataFrame
sentiment_df = pd.DataFrame(list(sentiment_series))

#  Merge back
sentiment_df.index = df.index

# Quick peek
sentiment_df.head()


100%|██████████| 5000/5000 [00:51<00:00, 97.53it/s] 


,polarity,subjectivity,label,raw_text,cleaned_text
0,-0.016667,0.379487,neutral,thought would big small paper turn like palm t...,thought would big small paper turn like palm t...
1,0.277778,0.844444,positive,kindle light easy use especially beach,kindle light easy use especially beach
2,0.165625,0.525000,positive,didnt know much id use kindle went lower end i...,didnt know much id use kindle went lower end i...
3,0.264062,0.534821,positive,happy purchase caught sale really good price n...,happy purchase caught sale really good price n...
4,0.464286,0.578571,positive,solid entry level kindle great kids gifted kid...,solid entry level kindle great kids gifted kid...


#Model testing

In [ ]:
# Testing model on sample reviews
sample_reviews = [
    "This product is amazing! The quality exceeded my expectations.",
    "Terrible experience. The item broke after one use.",
    "It's okay, not great but not terrible either.",
    "The product arrived late and was damaged.",
    "Love it. Works perfectly.",
]

for r in sample_reviews:
    print(analyze_sentiment(nlp, r, clean_fn=None))


{'polarity': 0.6000000000000001, 'subjectivity': 0.9, 'label': 'positive', 'raw_text': 'This product is amazing! The quality exceeded my expectations.', 'cleaned_text': 'product amazing quality exceeded expectations'}
{'polarity': -1.0, 'subjectivity': 1.0, 'label': 'negative', 'raw_text': 'Terrible experience. The item broke after one use.', 'cleaned_text': 'terrible experience item broke one use'}
{'polarity': 0.10000000000000002, 'subjectivity': 0.75, 'label': 'positive', 'raw_text': "It's okay, not great but not terrible either.", 'cleaned_text': 'okay great terrible either'}
{'polarity': -0.3, 'subjectivity': 0.6, 'label': 'negative', 'raw_text': 'The product arrived late and was damaged.', 'cleaned_text': 'product arrived late damaged'}
{'polarity': 0.75, 'subjectivity': 0.8, 'label': 'positive', 'raw_text': 'Love it. Works perfectly.', 'cleaned_text': 'love works perfectly'}


In [ ]:
# Testing edge cases
edge_cases = [
    "",                    # empty string
    "   ",                 # whitespace only
    None,                  # NoneType
    12345,                 # non-string
]

for text in edge_cases:
    result = analyze_sentiment(nlp, text)
    print(f"Input: {repr(text)}")
    print("Output:", result)
    print("-" * 70)


Input: ''
Output: {'polarity': 0.0, 'subjectivity': 0.0, 'label': 'neutral', 'raw_text': '', 'cleaned_text': ''}
----------------------------------------------------------------------
Input: '   '
Output: {'polarity': 0.0, 'subjectivity': 0.0, 'label': 'neutral', 'raw_text': '   ', 'cleaned_text': ''}
----------------------------------------------------------------------
Input: None
Output: {'polarity': 0.0, 'subjectivity': 0.0, 'label': 'neutral', 'raw_text': None, 'cleaned_text': ''}
----------------------------------------------------------------------
Input: 12345
Output: {'polarity': 0.0, 'subjectivity': 0.0, 'label': 'neutral', 'raw_text': 12345, 'cleaned_text': ''}
----------------------------------------------------------------------


In [ ]:
# Testing borderline review
borderline_reviews = [
    "It's fine, nothing special.",
    "Not bad, but not great either.",
    "I guess it works."
    "It was so bad, it does not work"
]

for text in borderline_reviews:
    result = analyze_sentiment(nlp, text, threshold_pos=0.2, threshold_neg=-0.2)
    print(f"Text: {text}")
    print(f"Polarity: {result['polarity']:.3f}, Label: {result['label']}")
    print("-" * 70)


Text: It's fine, nothing special.
Polarity: 0.387, Label: positive
----------------------------------------------------------------------
Text: Not bad, but not great either.
Polarity: 0.050, Label: neutral
----------------------------------------------------------------------
Text: I guess it works.It was so bad, it does not work
Polarity: -0.700, Label: negative
----------------------------------------------------------------------


In [ ]:
#Schema test
required_keys = {"polarity", "subjectivity", "label", "raw_text", "cleaned_text"}

sample_text = "Great value for money."
result = analyze_sentiment(nlp, sample_text)

assert set(result.keys()).issuperset(required_keys), "Missing required keys!"
assert isinstance(result["polarity"], float)
assert isinstance(result["subjectivity"], float)
assert result["label"] in {"positive", "neutral", "negative"}

print("Schema test passed ✔")


Schema test passed ✔


In [ ]:
# Cleaning text function test
sample_df = pd.DataFrame({
    "clean_text": [
        "Excellent product, very happy.",
        "Worst purchase ever.",
        "It works as described."
    ]
})

sentiment_df = sample_df["clean_text"].apply(
    lambda x: analyze_sentiment(nlp, x)
).apply(pd.Series)

sentiment_df


,polarity,subjectivity,label,raw_text,cleaned_text
0,0.9,1.0,positive,"Excellent product, very happy.",excellent product happy
1,-1.0,1.0,negative,Worst purchase ever.,worst purchase ever
2,0.0,0.0,neutral,It works as described.,works described


In [ ]:
# Model test on simple sample
text = "This product is a good product."

results = [analyze_sentiment(nlp, text)["label"] for _ in range(5)]
print(results)


['positive', 'positive', 'positive', 'positive', 'positive']


#Model application

In [ ]:
#Training data
train_df = pd.DataFrame({
    "clean_text": X_train,
    "true_label": label_encoder.inverse_transform(y_train)
})

train_df.head()


In [ ]:
train_sentiment = train_df["clean_text"].apply(
    lambda x: analyze_sentiment(nlp, x)
)

train_sentiment_df = pd.DataFrame(list(train_sentiment))
train_sentiment_df.index = train_df.index

# Combine predictions with train data
train_analysis_df = pd.concat(
    [train_df, train_sentiment_df[['polarity', 'subjectivity', 'label']]],
    axis=1
)

train_analysis_df.rename(columns={"label": "spacy_pred"}, inplace=True)
train_analysis_df.head()


In [ ]:
train_analysis_df.sample(10)


#Model evualution

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_true_train = label_encoder.transform(train_analysis_df["true_label"])
y_spacy_train = label_encoder.transform(train_analysis_df["spacy_pred"])

print("spaCy Analyzer – TRAIN sanity check accuracy:")
print(accuracy_score(y_true_train, y_spacy_train))

print("\nClassification Report (TRAIN sanity check):")
print(classification_report(
    y_true_train,
    y_spacy_train,
    target_names=label_encoder.classes_
))


In [ ]:
test_df = pd.DataFrame({
    "clean_text": X_test,
    "true_label": label_encoder.inverse_transform(y_test)
})

test_sentiment = test_df["clean_text"].apply(
    lambda x: analyze_sentiment(nlp, x)
)

test_sentiment_df = pd.DataFrame(list(test_sentiment))
test_sentiment_df.index = test_df.index

test_analysis_df = pd.concat(
    [test_df, test_sentiment_df[['polarity', 'subjectivity', 'label']]],
    axis=1
)

test_analysis_df.rename(columns={"label": "spacy_pred"}, inplace=True)
test_analysis_df.head()


In [ ]:
y_true_test = label_encoder.transform(test_analysis_df["true_label"])
y_spacy_test = label_encoder.transform(test_analysis_df["spacy_pred"])

print("spaCy Analyzer – TEST baseline accuracy:")
print(accuracy_score(y_true_test, y_spacy_test))

print("\nClassification Report (spaCy baseline):")
print(classification_report(
    y_true_test,
    y_spacy_test,
    target_names=label_encoder.classes_
))


Although the spaCy baseline achieved 79.9% accuracy, the dataset is highly imbalanced (93.7% positive samples). The model performs well on positive reviews (F1 = 0.89) but performs poorly on negative (F1 = 0.16) and neutral (F1 = 0.09) classes. The macro-average F1-score of 0.38 reveals weak general sentiment classification ability. Therefore, accuracy alone is misleading, and the model is biased toward the majority class.

In [ ]:
# Errors from test set
errors_df = test_analysis_df[
    test_analysis_df["true_label"] != test_analysis_df["spacy_pred"]
]

errors_df.sample(10)
